# Coil centering: parameter scan comparison

Compares the **old** multi-stage pipeline in `coil_centering_full.jl` (sinusoid + Fourier + local-B_phi initial guess → damped fixed-point bias correction) against the **new** minimal forward-model fit in `min_full.jl` (5D coordinate descent on the Hall B-field residual, starting from the baseline geometric pose).

Two scans:
1. **Pure x-shift** scan: `dx ∈ [0.01, 0.1, 1, 10, 100] mm`, no tilt.
2. **Pure tx-tilt** scan: `tx ∈ [0.01, 0.03, 0.1, 0.3, 1.0] deg`, no shift.

Both methods see the same baseline coil, the same synthetic perturbed coil, and the same Hall probe grid (sized to the baseline). Ground truth comes from the prescribed perturbation, not from any geometric fit.

In [9]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "../.."))

include("coil_centering_full.jl")
include("min_full.jl")

using Printf
using Plots
using Random
gr()

baseline_file = joinpath(@__DIR__, "sparc_pf1u.dat")
coil_current_a = 100.0

# Hall probe noise model (applied independently per component per probe):
#   σ_i = sqrt(noise_floor_T² + (noise_rel_frac · |B_clean|_i)²)
#
# Defaults model a realistic Hall probe:
#   - noise_floor_T  = 1.0e-5 T  =  0.1 Gauss absolute electronics floor
#   - noise_rel_frac = 1.0e-3    =  0.1 % gain/calibration error of |B|
#
# Set both to 0.0 for a noiseless scan. noise_seed is reapplied at the start
# of each scan loop so results are reproducible.
noise_floor_T  = 1.0e-5
noise_rel_frac = 1.0e-1
noise_seed     = 42

# Legacy uniform-noise knob (relative to max |B| across the grid). Kept at 0
# so only the realistic per-probe model above contributes.
noise_frac = 0.0

# Hall probe grid — shared by both methods, sized to baseline.
HALL_KWARGS = (
    hall_r_inner_frac        = 0.40,
    hall_r_outer_frac        = 0.50,
    hall_n_phi_inner         = 24,
    hall_n_z_inner           = 22,
    hall_n_phi_outer         = 20,
    hall_n_z_outer           = 10,
    hall_z_halfspan_inner_m  = 0.60,
    hall_z_halfspan_outer_m  = 0.40,
    add_noise_frac           = noise_frac,
    noise_floor_T            = noise_floor_T,
    noise_rel_frac           = noise_rel_frac,
)

baseline_coils = MinFull.load_coils(baseline_file; label="BASELINE", current_A=coil_current_a, verbose=false)
baseline_geom  = MinFull.axis_from_coil(baseline_coils; verbose=true, label="Baseline geometric axis")

  Activating project at `~/Documents/GitHub/JULIA_GPEC`
  Activating project at `~/Documents/GitHub/JULIA_GPEC`
  Activating 


Baseline geometric axis:


project at `~/Documents/GitHub/JULIA_GPEC`


  center = (-2.423, 1.082, 2307.411) mm
  tilt_x = 0.047918 deg, tilt_y = 0.102029 deg
  fit radius = 928.829 mm
  R range = [705.437, 1256.983] mm
  Z range = [2185.500, 3033.600] mm


(x0 = -0.002422875782886974, y0 = 0.0010819943116930845, z0 = 2.3074112258764488, tilt_x = 0.0479184987096367, tilt_y = 0.10202889002457728, tilt_mag = 0.11272123543606476, Rfit = 0.9288289996236926, Rmin = 0.705437494396208, Rmax = 1.256982811020103, Zmin = 2.1855, Zmax = 3.0336)

## Helper — run both methods on a programmatically perturbed coil

`scan_step` shifts/tilts the baseline coil by a prescribed `(dx, dy, dz, dtx, dty)`, builds Hall probes from the perturbed coil, then runs both pipelines on the same Hall data.

- **New method**: `MinFull.fit_pose_to_hall(baseline_coils, baseline_geom, hall)` — 5D coordinate descent.
- **Old method**: `magnetic_z_tilt` (both shells) → `magnetic_xy_minimize` (both shells) → `bias_correct_axis(baseline_coils, hall, zt1, zt2, sin1, sin2)`. `bias_correct_axis` prints per-iteration logs, so we redirect its stdout.

Both methods return absolute poses `(x0, y0, z0, tilt_x, tilt_y)`. We convert to deltas vs. the baseline geometric pose for comparison against the prescribed perturbation.

In [10]:
function scan_step(baseline_coils, baseline_geom, dx, dy, dz, dtx, dty;
        step0_m=0.050, step0_deg=0.5, tol_m=1e-7, tol_deg=1e-5, max_iter=300,
        n_bias_iter=4, bias_damping=0.6)

    # Build perturbed coil by rigid shift + tilt of the baseline
    perturbed_coils = MinFull.place_coil_at_axis(baseline_coils,
        baseline_geom.x0 + dx,
        baseline_geom.y0 + dy,
        baseline_geom.z0 + dz,
        baseline_geom.tilt_x + dtx,
        baseline_geom.tilt_y + dty)

    # Shared Hall data — perturbed coil is the source, grid sized to baseline
    hall = MinFull.make_hall_data(perturbed_coils, baseline_geom; verbose=false, HALL_KWARGS...)

    # New method
    t_new = @elapsed new_pose = MinFull.fit_pose_to_hall(baseline_coils, baseline_geom, hall;
        step0_m=step0_m, step0_deg=step0_deg, tol_m=tol_m, tol_deg=tol_deg,
        max_iter=max_iter, verbose=false)

    # Old method: sinusoid Z+tilt → local-B_phi XY → damped fixed-point bias correction
    t_old = @elapsed old_pose = redirect_stdout(devnull) do
        zt1  = magnetic_z_tilt(hall, baseline_geom; shell=1, verbose=false)
        zt2  = magnetic_z_tilt(hall, baseline_geom; shell=2, verbose=false)
        sin1 = magnetic_xy_minimize(hall, baseline_geom, zt1; shell=1, verbose=false)
        sin2 = magnetic_xy_minimize(hall, baseline_geom, zt2; shell=2, verbose=false)
        bias_correct_axis(baseline_coils, hall, zt1, zt2, sin1, sin2;
            n_iter=n_bias_iter, damping=bias_damping)
    end

    delta(a) = (
        dx  = (a.x0 - baseline_geom.x0)*1e3,
        dy  = (a.y0 - baseline_geom.y0)*1e3,
        dz  = (a.z0 - baseline_geom.z0)*1e3,
        dtx = a.tilt_x - baseline_geom.tilt_x,
        dty = a.tilt_y - baseline_geom.tilt_y,
    )

    return (
        truth = (dx=dx*1e3, dy=dy*1e3, dz=dz*1e3, dtx=dtx, dty=dty),
        new   = delta(new_pose),
        old   = delta(old_pose),
        new_objective = new_pose.objective,
        new_n_eval    = new_pose.n_eval,
        t_new = t_new,
        t_old = t_old,
    )
end

scan_step (generic function with 1 method)

## Shift scan: `dx ∈ [0.01, 0.1, 1, 10, 100] mm`

Pure x-shift. The recovered Δx should equal the prescribed Δx; Δy, Δz, Δtx, Δty should all be zero.

In [11]:
shifts_mm   = [0.01, 0.1, 1.0, 10.0, 100.0]
shift_results = Vector{NamedTuple}(undef, length(shifts_mm))

Random.seed!(noise_seed)   # reproducible noise realization across the scan

println("\nShift scan (dx applied as pure x-translation):")
@printf("  noise floor = %.2g T (%.3g G), rel error = %.2f%%\n",
        noise_floor_T, noise_floor_T*1e4, noise_rel_frac*100)
@printf("%-12s | %-30s | %-30s | %-12s\n", "true dx", "new (dx, |dy|, |dz|)", "old (dx, |dy|, |dz|)", "speed new/old")
println("-" ^ 100)
for (i, dx_mm) in enumerate(shifts_mm)
    r = scan_step(baseline_coils, baseline_geom, dx_mm*1e-3, 0.0, 0.0, 0.0, 0.0;
        step0_m=max(0.050, 2*dx_mm*1e-3), step0_deg=0.5)
    shift_results[i] = r
    @printf("%9.3f mm | %+8.4f, %.4f, %.4f mm | %+8.4f, %.4f, %.4f mm | %.2fs / %.2fs\n",
            dx_mm,
            r.new.dx, abs(r.new.dy), abs(r.new.dz),
            r.old.dx, abs(r.old.dy), abs(r.old.dz),
            r.t_new, r.t_old)
end


Shift scan (dx applied as pure x-translation):
  noise floor = 1e-05 T (0.1 G), rel error = 10.00%
true dx      | new (dx, |dy|, |dz|)           | old (dx, |dy|, |dz|)           | speed new/old
----------------------------------------------------------------------------------------------------
    0.010 mm |  -2.4143, 6.7423, 1.0757 mm |  -6.8227, 5.9620, 1.6556 mm | 20.45s / 2.01s
    0.100 mm |  -3.4386, 5.0846, 1.5442 mm |  -0.7663, 12.7060, 3.4047 mm | 17.16s / 0.29s
    1.000 mm |  -4.1964, 1.8803, 2.0300 mm |  -8.7068, 4.8019, 14.1893 mm | 17.87s / 0.26s
   10.000 mm |  +9.8118, 2.8009, 0.6355 mm |  +7.7334, 7.5778, 9.3442 mm | 18.24s / 0.25s
  100.000 mm | +103.0499, 2.2150, 5.3707 mm | +122.0921, 21.8549, 7.3944 mm | 49.57s / 0.27s


In [12]:
abs_err_floor(v) = max(abs(v), 1e-12)

true_dx       = shifts_mm
new_dx_err    = [abs_err_floor(r.new.dx - r.truth.dx)            for r in shift_results]
old_dx_err    = [abs_err_floor(r.old.dx - r.truth.dx)            for r in shift_results]
new_other_err = [abs_err_floor(hypot(r.new.dy, r.new.dz))        for r in shift_results]
old_other_err = [abs_err_floor(hypot(r.old.dy, r.old.dz))        for r in shift_results]
new_tilt_err  = [abs_err_floor(hypot(r.new.dtx, r.new.dty))      for r in shift_results]
old_tilt_err  = [abs_err_floor(hypot(r.old.dtx, r.old.dty))      for r in shift_results]

p_shift = plot(layout=(1,2), size=(1100, 420), left_margin=8Plots.mm, bottom_margin=6Plots.mm)

plot!(p_shift[1], true_dx, new_dx_err; xaxis=:log10, yaxis=:log10, marker=:circle,
      label="new |Δx error|", lw=2)
plot!(p_shift[1], true_dx, old_dx_err; marker=:square, label="old |Δx error|", lw=2)
plot!(p_shift[1], true_dx, true_dx ./ 100; ls=:dash, color=:gray, label="1% of true")
xlabel!(p_shift[1], "true Δx [mm]")
ylabel!(p_shift[1], "|recovered − true| [mm]")
title!(p_shift[1], "Shift scan: in-axis (Δx) error")

plot!(p_shift[2], true_dx, new_other_err; xaxis=:log10, yaxis=:log10, marker=:circle,
      label="new |Δy,Δz| spurious", lw=2)
plot!(p_shift[2], true_dx, old_other_err; marker=:square, label="old |Δy,Δz| spurious", lw=2)
plot!(p_shift[2], true_dx, new_tilt_err; marker=:utriangle, label="new |tilt| spurious [deg]", lw=2)
plot!(p_shift[2], true_dx, old_tilt_err; marker=:dtriangle, label="old |tilt| spurious [deg]", lw=2)
xlabel!(p_shift[2], "true Δx [mm]")
ylabel!(p_shift[2], "|spurious component|")
title!(p_shift[2], "Shift scan: cross-axis leakage")

savefig(p_shift, joinpath(@__DIR__, "comp_min_shift_scan.png"))
println("Saved: ", joinpath(@__DIR__, "comp_min_shift_scan.png"))
p_shift

UndefVarError: UndefVarError: `plot` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in GR.jlgr.
    - Also exported by GR.
Hint: a global variable of this name also exists in RecipesBase.
    - Also exported by Plots.
Hint: a global variable of this name also exists in Makie.
    - Also exported by GLMakie.

## Tilt scan: `tx ∈ [0.01, 0.03, 0.1, 0.3, 1.0] deg`

Pure tx-tilt. The recovered Δtx should equal the prescribed Δtx; Δty, Δx, Δy, Δz should all be zero (tilt is applied about the coil's geometric center, so center stays put).

In [13]:
tilts_deg = [0.01, 0.03, 0.1, 0.3, 1.0]
tilt_results = Vector{NamedTuple}(undef, length(tilts_deg))

Random.seed!(noise_seed)   # reproducible noise realization across the scan

println("\nTilt scan (dtx applied as pure tx-tilt):")
@printf("  noise floor = %.2g T (%.3g G), rel error = %.2f%%\n",
        noise_floor_T, noise_floor_T*1e4, noise_rel_frac*100)
@printf("%-13s | %-30s | %-30s | %-12s\n", "true dtx", "new (dtx, |dty|, |dx|mm)", "old (dtx, |dty|, |dx|mm)", "speed new/old")
println("-" ^ 105)
for (i, dtx_deg) in enumerate(tilts_deg)
    r = scan_step(baseline_coils, baseline_geom, 0.0, 0.0, 0.0, dtx_deg, 0.0;
        step0_m=0.050, step0_deg=max(0.5, 2*dtx_deg))
    tilt_results[i] = r
    @printf("%9.4f deg | %+8.5f, %.5f, %.4f    | %+8.5f, %.5f, %.4f    | %.2fs / %.2fs\n",
            dtx_deg,
            r.new.dtx, abs(r.new.dty), abs(r.new.dx),
            r.old.dtx, abs(r.old.dty), abs(r.old.dx),
            r.t_new, r.t_old)
end


Tilt scan (dtx applied as pure tx-tilt):
  noise floor = 1e-05 T (0.1 G), rel error = 10.00%
true dtx      | new (dtx, |dty|, |dx|mm)       | old (dtx, |dty|, |dx|mm)       | speed new/old
---------------------------------------------------------------------------------------------------------
   0.0100 deg | -0.46618, 0.35171, 2.4231    | -0.31790, 0.04423, 6.8307    | 19.28s / 0.32s
   0.0300 deg | +0.07478, 0.05573, 3.5395    | +0.47101, 0.03205, 0.8574    | 17.49s / 0.27s
   0.1000 deg | +0.30375, 0.09603, 5.1525    | +0.37239, 0.28758, 9.3825    | 18.46s / 0.27s
   0.3000 deg | +0.00601, 0.13549, 1.0847    | -0.37303, 0.48051, 2.2461    | 16.40s / 0.24s
   1.0000 deg | +1.27158, 0.26079, 2.6814    | +0.96327, 0.00452, 18.1632    | 18.56s / 0.26s


In [14]:
true_dtx       = tilts_deg
new_dtx_err    = [abs_err_floor(r.new.dtx - r.truth.dtx)         for r in tilt_results]
old_dtx_err    = [abs_err_floor(r.old.dtx - r.truth.dtx)         for r in tilt_results]
new_dty_err    = [abs_err_floor(r.new.dty)                       for r in tilt_results]
old_dty_err    = [abs_err_floor(r.old.dty)                       for r in tilt_results]
new_shift_err  = [abs_err_floor(hypot(r.new.dx, r.new.dy, r.new.dz)) for r in tilt_results]
old_shift_err  = [abs_err_floor(hypot(r.old.dx, r.old.dy, r.old.dz)) for r in tilt_results]

p_tilt = plot(layout=(1,2), size=(1100, 420), left_margin=8Plots.mm, bottom_margin=6Plots.mm)

plot!(p_tilt[1], true_dtx, new_dtx_err; xaxis=:log10, yaxis=:log10, marker=:circle,
      label="new |Δtx error|", lw=2)
plot!(p_tilt[1], true_dtx, old_dtx_err; marker=:square, label="old |Δtx error|", lw=2)
plot!(p_tilt[1], true_dtx, true_dtx ./ 100; ls=:dash, color=:gray, label="1% of true")
xlabel!(p_tilt[1], "true Δtx [deg]")
ylabel!(p_tilt[1], "|recovered − true| [deg]")
title!(p_tilt[1], "Tilt scan: in-axis (Δtx) error")

plot!(p_tilt[2], true_dtx, new_dty_err;   xaxis=:log10, yaxis=:log10, marker=:circle,    label="new |Δty| spurious [deg]", lw=2)
plot!(p_tilt[2], true_dtx, old_dty_err;   marker=:square,    label="old |Δty| spurious [deg]",   lw=2)
plot!(p_tilt[2], true_dtx, new_shift_err; marker=:utriangle, label="new |shift| spurious [mm]",  lw=2)
plot!(p_tilt[2], true_dtx, old_shift_err; marker=:dtriangle, label="old |shift| spurious [mm]",  lw=2)
xlabel!(p_tilt[2], "true Δtx [deg]")
ylabel!(p_tilt[2], "|spurious component|")
title!(p_tilt[2], "Tilt scan: cross-component leakage")

savefig(p_tilt, joinpath(@__DIR__, "comp_min_tilt_scan.png"))
println("Saved: ", joinpath(@__DIR__, "comp_min_tilt_scan.png"))
p_tilt

UndefVarError: UndefVarError: `plot` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.
Hint: a global variable of this name also exists in GR.jlgr.
    - Also exported by GR.
Hint: a global variable of this name also exists in RecipesBase.
    - Also exported by Plots.
Hint: a global variable of this name also exists in Makie.
    - Also exported by GLMakie.

## Summary table — relative error and runtime

`rel_err = |recovered − true| / true`. Lower is better. Runtime is wall-clock for each individual fit.

In [15]:
println("\n=== SHIFT SCAN SUMMARY ===")
@printf("%-12s | %-14s | %-14s | %-14s | %-14s\n",
        "true dx [mm]", "new rel.err", "old rel.err", "new time [s]", "old time [s]")
println("-" ^ 80)
for (dx_mm, r) in zip(shifts_mm, shift_results)
    new_re = abs(r.new.dx - r.truth.dx) / abs(r.truth.dx)
    old_re = abs(r.old.dx - r.truth.dx) / abs(r.truth.dx)
    @printf("%12.3f | %14.3e | %14.3e | %14.2f | %14.2f\n",
            dx_mm, new_re, old_re, r.t_new, r.t_old)
end

println("\n=== TILT SCAN SUMMARY ===")
@printf("%-13s | %-14s | %-14s | %-14s | %-14s\n",
        "true dtx [deg]", "new rel.err", "old rel.err", "new time [s]", "old time [s]")
println("-" ^ 80)
for (dtx_deg, r) in zip(tilts_deg, tilt_results)
    new_re = abs(r.new.dtx - r.truth.dtx) / abs(r.truth.dtx)
    old_re = abs(r.old.dtx - r.truth.dtx) / abs(r.truth.dtx)
    @printf("%13.4f | %14.3e | %14.3e | %14.2f | %14.2f\n",
            dtx_deg, new_re, old_re, r.t_new, r.t_old)
end


=== SHIFT SCAN SUMMARY ===
true dx [mm] | new rel.err    | old rel.err    | new time [s]   | old time [s]  
--------------------------------------------------------------------------------
       0.010 |      2.424e+02 |      6.833e+02 |          20.45 |           2.01
       0.100 |      3.539e+01 |      8.663e+00 |          17.16 |           0.29
       1.000 |      5.196e+00 |      9.707e+00 |          17.87 |           0.26
      10.000 |      1.882e-02 |      2.267e-01 |          18.24 |           0.25
     100.000 |      3.050e-02 |      2.209e-01 |          49.57 |           0.27

=== TILT SCAN SUMMARY ===
true dtx [deg] | new rel.err    | old rel.err    | new time [s]   | old time [s]  
--------------------------------------------------------------------------------
       0.0100 |      4.762e+01 |      3.279e+01 |          19.28 |           0.32
       0.0300 |      1.493e+00 |      1.470e+01 |          17.49 |           0.27
       0.1000 |      2.037e+00 |      2.724e+00 | 